In [1]:
import os
import multiprocessing as mp
import numpy as np
import tiktoken
from datasets import load_dataset
from tqdm import tqdm
from functools import partial

In [2]:
ds = load_dataset("yahma/alpaca-cleaned")

README.md:   0%|          | 0.00/11.6k [00:00<?, ?B/s]

alpaca_data_cleaned.json:   0%|          | 0.00/44.3M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/51760 [00:00<?, ? examples/s]

In [14]:
def prompt_no_input(row):
    return ("Below is an instruction that describes a task. "
            "Write a response that appropriately completes the request.\n\n"
            "### Instruction:\n{instruction}\n\n### Response:\n").format_map(row)


def prompt_input(row):
    return ("Below is an instruction that describes a task, paired with an input that provides further context. "
            "Write a response that appropriately completes the request.\n\n"
            "### Instruction:\n{instruction}\n\n### Input:\n{input}\n\n### Response:\n").format_map(row)

def create_prompt(row):
    return prompt_no_input(row) if row["input"] == "" else prompt_input(row)

def tokenize(doc, enc, eot):
    # tokenizes a single document and returns a numpy array of uint16 tokens
    tokens = [eot]
    prompt = create_prompt(doc)
    output = doc["output"]
    full_text = prompt + output
    tokens.extend(enc.encode_ordinary(full_text))
    tokens_np = np.array(tokens)
    assert (0 <= tokens_np).all() and (tokens_np < 2**16).all(), "tokens must be uint16"
    tokens_np = tokens_np.astype(np.uint16)
    return tokens_np

In [18]:
enc = tiktoken.encoding_for_model("gpt2")
eot = enc.eot_token

In [26]:
# calculate total number of tokens in the dataset
total_tokens = 0
for doc in ds["train"]:
    total_tokens += len(tokenize(doc, enc, eot))

print(f"Total tokens in million: {total_tokens / 1e6:.2f} M")

Total tokens in million: 10.43 M


- We can keep aside 10% for validation and 10% for testing

In [27]:
train_size = total_tokens * 0.8
val_size = total_tokens * 0.1
test_size = total_tokens * 0.1

In [28]:
def split_and_save_tokens(ds, total_tokens, enc, eot):
    # Calculate split sizes based on percentages
    test_size = int(0.1 * total_tokens)   # 10% for test
    val_size = int(0.1 * total_tokens)    # 10% for validation
    
    # Initialize lists to collect tokens for each split
    test_tokens = []
    val_tokens = []
    train_tokens = []
    
    # Set up multiprocessing
    nprocs = max(1, mp.cpu_count() // 2)
    partial_tokenize = partial(tokenize, enc=enc, eot=eot)
    
    with mp.Pool(nprocs) as pool:
        token_count = 0
        progress_bar = tqdm(total=total_tokens, desc="Processing tokens")
        
        for tokens in pool.imap(partial_tokenize, ds["train"], chunksize=10):
            current_tokens = len(tokens)
            
            # Add tokens to appropriate split
            if token_count + current_tokens <= test_size:
                test_tokens.append(tokens)
            elif token_count + current_tokens <= test_size + val_size:
                val_tokens.append(tokens)
            else:
                train_tokens.append(tokens)
            
            # Update counters and progress
            token_count += current_tokens
            progress_bar.update(current_tokens)
            
            # Check if we've collected enough tokens
            if token_count >= total_tokens:
                break
        
        progress_bar.close()
    
    # Convert lists to numpy arrays and save
    print("\nSaving split datasets...")
    
    test_array = np.array(test_tokens, dtype=np.int32)
    np.save(os.path.join("data", "test.npy"), test_array)
    print(f"Test set saved: {len(test_array):,} tokens")
    
    val_array = np.array(val_tokens, dtype=np.int32)
    np.save(os.path.join("data", "validation.npy"), val_array)
    print(f"Validation set saved: {len(val_array):,} tokens")
    
    train_array = np.array(train_tokens, dtype=np.int32)
    np.save(os.path.join("data", "train.npy"), train_array)
    print(f"Training set saved: {len(train_array):,} tokens")
    
    print(f"\nTotal processed: {token_count:,} tokens")

In [29]:
split_and_save_tokens(ds, total_tokens, enc, eot)

Processing tokens:   0%|          | 0/10433055 [00:00<?, ?it/s]Process SpawnPoolWorker-3:
Process SpawnPoolWorker-4:
Process SpawnPoolWorker-2:
Process SpawnPoolWorker-5:
Process SpawnPoolWorker-1:
Traceback (most recent call last):
Traceback (most recent call last):
Traceback (most recent call last):
Traceback (most recent call last):
Traceback (most recent call last):
  File "/opt/miniconda3/envs/transformer-proj/lib/python3.12/multiprocessing/process.py", line 314, in _bootstrap
    self.run()
  File "/opt/miniconda3/envs/transformer-proj/lib/python3.12/multiprocessing/process.py", line 314, in _bootstrap
    self.run()
  File "/opt/miniconda3/envs/transformer-proj/lib/python3.12/multiprocessing/process.py", line 108, in run
    self._target(*self._args, **self._kwargs)
  File "/opt/miniconda3/envs/transformer-proj/lib/python3.12/multiprocessing/process.py", line 108, in run
    self._target(*self._args, **self._kwargs)
  File "/opt/miniconda3/envs/transformer-proj/lib/python3.12/mu

KeyboardInterrupt: 